# Step-by-step Guide to Installing Pytorch-GPU on Mac Pro Max `M3` Silicon Chip

**Author: Dr. Saad Laouadi**

**Date: August 24th, 2024**

# Table of Contents

1. [Objective](#Objective)
2. [Introduction](#Introduction)
3. [Metal Acceleration for PyTorch on Apple Silicon (M1, M2, M3)](#Metal-Acceleration-for-PyTorch-on-Apple-Silicon-(M1,-M2,-M3))
4. [Setup Steps Using Conda Package Manager](#Setup-Steps-Using-Conda-Package-Manager)
5. [JupyterLab Setup](#JupyterLab-Setup)
6. [Test the Environment](#Test-the-Environment)
7. [GPU Vs CPU Testing Using Small Dataset](#GPU-Vs-CPU-Testing-Using-Small-Dataset)
8. [GPU Vs CPU Testing Using Large Dataset](#GPU-Vs-CPU-Testing-Using-Large-Dataset)
9. [Supplementary Information](#Supplementary-Information)

## Objective

The main objective of this notebooks is:

1. **Provide a comprehensive, step-by-step guide for installing PyTorch-GPU on Mac M3 Max Silicon Chip**2.
2. **Ensure compatibility between PyTorch and the M3 Max architecture**
3. **Optimize PyTorch performance by leveraging GPU acceleration on M3 Max**
4. **Help users overcome potential challenges in the installation process**
5. **Enable machine learning practitioners to use PyTorch efficiently on the latest Mac hardware**

## Introduction

In this notebook, we will guide you through the process of installing **PyTorch** on a MacBook Pro with the M3 Max chip, featuring Apple's Silicon architecture. We will begin by setting up **Miniforge**, a minimal installer for **Conda** and **Mamba**. Following that, we will create virtual environments using both the `conda` (via `mamba`) package manager. In the next notebook we will use  `pip3` to do similar steps.

> **Note**: Throughout this tutorial, we will primarily use `mamba` for most commands.

To demonstrate the effectiveness of the setup, we will run code on both the `GPU` and `CPU`, and train a deep learning model to compare performance and runtime on these devices.

## Metal Acceleration for PyTorch on Apple Silicon (M1, M2, M3)

PyTorch supports GPU acceleration on Macs with Apple Silicon (M1, M2, M3) using the Metal Performance Shaders (MPS) backend. This MPS backend is optimized for Apple Silicon, enabling efficient GPU-based training directly on your Mac.

The MPS backend takes full advantage of the unique architecture of Apple Silicon, utilizing finely-tuned kernels to maximize performance. With the **mps** device, PyTorch can seamlessly map deep learning tasks to the MPS Graph framework, providing significant acceleration for training models on Macs with M1, M2, or M3 chips.

## Setup Steps Using `Conda` Package Manager

1. **Install Homebrew**: If you haven’t installed Homebrew, you can do so by running the following command in your terminal:

   ```sh
   /bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"
   ```

2. **Install Miniforge**: Miniforge is a minimal conda installer that is compatible with M3 Macs.


   ```sh
   brew install --cask miniforge
   ```

If you prefer to use the graphical installer of **Miniforge**, you can download it for here [link](https://conda-forge.org/download/)

4. **Create a Conda Environment**: Create a conda environment for  using the `mamba` package manager:

    ```sh
    mamba create --name torch-env python=3.10 --yes
    ```
    
5. **Activate the newly created environment**:

   ```sh
   mamba activate torch-env
   ```

6. **Install the latest version of Pytorch**:  The next command will install only **pytorch**

```sh
mamba install pytorch -c pytorch-nightly --yes
```

It depends on the specific projects you’re working on. If your work involves images, text, or audio, you can install the following PyTorch-related libraries

   ```sh
   mamba install torchvision torchtext torchaudio -c pytorch-nightly --yes
   ```

That is it, **Pytorch** is installed and ready to be tested. 

7. **Check the installation**:

    ```sh
    pip list | grep -E 'torch'
    ```
    
**Here is the output on my machine:**

```text
torch              2.4.0
torchaudio         2.4.0
torchtext          0.6.0
torchvision        0.19.0
```

## JupyterLab Setup 

1. **Install JupyterLab web-based application**:

   ```sh
   python3 -m pip install jupyterlab
   ```

2. **Install JupyterLab Desktop application**: Alternatively, you can install the JupyterLab desktop application (if you don't already have that):

    
    ```sh
    brew install --cask jupyterlab
    ```
    
3. **Register the kernel to be available for JupyterLab**:

    ```sh
    mamba install ipykernel
    ipython kernel install --user --name=torch-env --display-name "Torch-GPU Py3.10"
    ```
    
## Test the Environment

1. **Create a working directory for a new project**:

    ```sh
    mkdir test-project && cd test-project
    ```
    
2. **Open JupyterLab**:

- Web-based:

   ```sh
   jupyter lab .
   ```

- Desktop JupyterLab:

   ```sh
   open -a JupyterLab . 
   ```

In [1]:
#===========================================================================#
#             Testing Pytorch on Mac M3 Max 
#===========================================================================#

import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# Check if the current Python executable matches the expected path
expected_executable = "/opt/homebrew/Caskroom/miniforge/base/envs/torch-env/bin/python3.10"
assert sys.executable == expected_executable, f"Expected Python executable: {expected_executable}, but got: {sys.executable}"

# Check torch version (2.4)
assert torch.__version__.startswith('2.4'), f"Expected torch version to start with '2.4', but got: {torch.__version__}"

# Print system and Pytorch version information
print("="*72)
print(f'System: {sys.executable}')
print(f'Pytorch: {torch.__version__}')
print("="*72)

System: /opt/homebrew/Caskroom/miniforge/base/envs/torch-env/bin/python3.10
Pytorch: 2.4.0


In [2]:
# Test for the available GPU (MPS)
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


2024-08-24 00:20:36.189 python3.10[35518:511553] Metal API Validation Enabled


## GPU Vs. CPU Testing: Generating Data

In [3]:
%timeit

def test_device(device):
    print(f"\nTesting on {device}...")
    
    data_size = (10000, 10000)
    start_time = time.time()
    
    x = torch.randn(data_size, device=device)
    y = torch.randn(data_size, device=device)

    z = torch.matmul(x, y)
    
    result = z.sum().item()
    
    end_time = time.time()
    
    print(f"Result: {result}")
    print(f"Time taken on {device}: {end_time - start_time} seconds")
    
# Check if MPS device is available
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    test_device(mps_device)
else:
    print("MPS device not found.")
    
# Test on CPU
cpu_device = torch.device("cpu")
test_device(cpu_device)


Testing on mps...
Result: 991771.5
Time taken on mps: 0.27329516410827637 seconds

Testing on cpu...
Result: 1120241.125
Time taken on cpu: 2.503387928009033 seconds


## GPU Vs CPU Testing Using Small Dataset

1. **Using GPU**

In [4]:
%%time

N_EPOCHS = 5

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
 
print(f"Using device: {device}")

# Simple neural network model
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 28 * 28)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Define the transformation 
transform = transforms.Compose([
    transforms.Resize((28, 28)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.2860,), std=(0.3530,))
])

# Loading the FashionMNIST dataset with the defined transformations
train_dataset = datasets.FashionMNIST(root='./data',
                                      train=True,
                                      download=True,
                                      transform=transform)

test_dataset = datasets.FashionMNIST(root='./data',
                                     train=False,
                                     download=True,
                                     transform=transform)

# Creating data loaders for training and testing datasets
train_loader = DataLoader(dataset=train_dataset,
                          batch_size=64,
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=64,
                         shuffle=True)

# Instantiate the model, loss function, and optimizer
model = SimpleNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training the model
for epoch in range(1, N_EPOCHS):  
    model.train()
    print(f'Epoch -------------- {epoch} ------------------')
    total_step = len(train_loader)
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)  
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
             print(f'Epoch [{epoch}/{N_EPOCHS}], Step [{batch_idx + 1}/{total_step}], Loss: {loss.item():.4f}')

    print(f'Epoch {epoch} completed.\n')

print("Training completed.")

Using device: mps
Epoch -------------- 1 ------------------
Epoch [1/5], Step [1/938], Loss: 2.2914
Epoch [1/5], Step [101/938], Loss: 0.5038
Epoch [1/5], Step [201/938], Loss: 0.3724
Epoch [1/5], Step [301/938], Loss: 0.4092
Epoch [1/5], Step [401/938], Loss: 0.5208
Epoch [1/5], Step [501/938], Loss: 0.3630
Epoch [1/5], Step [601/938], Loss: 0.3411
Epoch [1/5], Step [701/938], Loss: 0.4693
Epoch [1/5], Step [801/938], Loss: 0.3119
Epoch [1/5], Step [901/938], Loss: 0.3171
Epoch 1 completed.

Epoch -------------- 2 ------------------
Epoch [2/5], Step [1/938], Loss: 0.2665
Epoch [2/5], Step [101/938], Loss: 0.2512
Epoch [2/5], Step [201/938], Loss: 0.4353
Epoch [2/5], Step [301/938], Loss: 0.2966
Epoch [2/5], Step [401/938], Loss: 0.3098
Epoch [2/5], Step [501/938], Loss: 0.3263
Epoch [2/5], Step [601/938], Loss: 0.2342
Epoch [2/5], Step [701/938], Loss: 0.2940
Epoch [2/5], Step [801/938], Loss: 0.3389
Epoch [2/5], Step [901/938], Loss: 0.3134
Epoch 2 completed.

Epoch -------------- 3

In [5]:
# Model Evaluation
model.eval()

# prediction
test_loss = 0
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = test_loss / len(test_loader)

    print(f'Accuracy of the network on the test images: {accuracy:.2f} %')
    print(f'Average loss on the test set: {avg_loss:.4f}')

Accuracy of the network on the test images: 86.81 %
Average loss on the test set: 0.3641


2. **Testing on CPU**

In [6]:
%%time

# Set to use `CPU`
device = torch.device("cpu")
 
print(f"Using device: {device}")

model = SimpleNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training the model on CPU
for epoch in range(1, N_EPOCHS):  
    model.train()
    print(f'Epoch -------------- {epoch} ------------------')
    total_step = len(train_loader)
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)  
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
             print(f'Epoch [{epoch}/{N_EPOCHS}], Step [{batch_idx + 1}/{total_step}], Loss: {loss.item():.4f}')

    print(f'Epoch {epoch} completed.\n')

Using device: cpu
Epoch -------------- 1 ------------------
Epoch [1/5], Step [1/938], Loss: 2.3519
Epoch [1/5], Step [101/938], Loss: 0.4405
Epoch [1/5], Step [201/938], Loss: 0.5070
Epoch [1/5], Step [301/938], Loss: 0.3813
Epoch [1/5], Step [401/938], Loss: 0.3820
Epoch [1/5], Step [501/938], Loss: 0.5208
Epoch [1/5], Step [601/938], Loss: 0.5587
Epoch [1/5], Step [701/938], Loss: 0.3433
Epoch [1/5], Step [801/938], Loss: 0.2987
Epoch [1/5], Step [901/938], Loss: 0.2709
Epoch 1 completed.

Epoch -------------- 2 ------------------
Epoch [2/5], Step [1/938], Loss: 0.6411
Epoch [2/5], Step [101/938], Loss: 0.3820
Epoch [2/5], Step [201/938], Loss: 0.4329
Epoch [2/5], Step [301/938], Loss: 0.3872
Epoch [2/5], Step [401/938], Loss: 0.3405
Epoch [2/5], Step [501/938], Loss: 0.3709
Epoch [2/5], Step [601/938], Loss: 0.2395
Epoch [2/5], Step [701/938], Loss: 0.4265
Epoch [2/5], Step [801/938], Loss: 0.4372
Epoch [2/5], Step [901/938], Loss: 0.3117
Epoch 2 completed.

Epoch -------------- 3

In [7]:
# Model Evaluation
model.eval()

# prediction
test_loss = 0
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = test_loss / len(test_loader)

    print(f'Accuracy of the network on the test images: {accuracy:.2f} %')
    print(f'Average loss on the test set: {avg_loss:.4f}')

Accuracy of the network on the test images: 87.78 %
Average loss on the test set: 0.3456


## GPU Vs CPU Testing Using Large Dataset

1. **Testing on GPU**

In [8]:
%%time

N_EPOCHS = 4                             # change this for more epochs

# Set the device to GPU if available, otherwise CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(f"Using device: {device}")

# Define the transformation for the CIFAR-100 dataset
transform = transforms.Compose([
    transforms.Resize((32, 32)),  
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5071, 0.4867, 0.4408), std=(0.2675, 0.2565, 0.2761))
])

# Loading the CIFAR-100 dataset with the defined transformations
train_dataset = datasets.CIFAR100(root='./data',
                                  train=True,
                                  download=True,
                                  transform=transform)

test_dataset = datasets.CIFAR100(root='./data',
                                 train=False,
                                 download=True,
                                 transform=transform)

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=128,
                          shuffle=True,
                          num_workers=4)

# Load the ResNet50 model
model = models.resnet50(num_classes=100).to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training the model
for epoch in range(1, N_EPOCHS):  
    model.train()
    print(f'Epoch -------------- {epoch} ------------------')
    total_step = len(train_loader)
    
    # Start timing for the epoch
    start_time = time.time()
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)  
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        if batch_idx % 100 == 0:
            print(f'Epoch [{epoch}/3], Step [{batch_idx + 1}/{total_step}], Loss: {loss.item():.4f}')
    
    # Calculate and print the time taken for the epoch
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Epoch {epoch} completed in: {elapsed_time:.2f} seconds on {device}.\n")

print("Training completed.")

Using device: mps
Files already downloaded and verified
Files already downloaded and verified
Epoch -------------- 1 ------------------
Epoch [1/3], Step [1/391], Loss: 5.0069
Epoch [1/3], Step [101/391], Loss: 4.7126
Epoch [1/3], Step [201/391], Loss: 4.4702
Epoch [1/3], Step [301/391], Loss: 3.5812
Epoch 1 completed in: 50.44 seconds on mps.

Epoch -------------- 2 ------------------
Epoch [2/3], Step [1/391], Loss: 3.4344
Epoch [2/3], Step [101/391], Loss: 3.2897
Epoch [2/3], Step [201/391], Loss: 3.2286
Epoch [2/3], Step [301/391], Loss: 3.5277
Epoch 2 completed in: 49.69 seconds on mps.

Epoch -------------- 3 ------------------
Epoch [3/3], Step [1/391], Loss: 3.0287
Epoch [3/3], Step [101/391], Loss: 2.8758
Epoch [3/3], Step [201/391], Loss: 3.5961
Epoch [3/3], Step [301/391], Loss: 3.4881
Epoch 3 completed in: 49.64 seconds on mps.

Training completed.
CPU times: user 1min 19s, sys: 9.19 s, total: 1min 28s
Wall time: 2min 31s


In [11]:
# Evaluation of the ResNet50 model on the CIFAR-100 test set
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=128,
                         shuffle=False,
                         num_workers=12)

model.eval()
correct = 0
total = 0
test_loss = 0.0

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        
        outputs = model(data)
        loss = criterion(outputs, target)
        test_loss += loss.item() * data.size(0)

        _, predicted = torch.max(outputs.data, 1)
        
        total += target.size(0)
        correct += (predicted == target).sum().item()

test_loss /= len(test_loader.dataset)
accuracy = 100 * correct / total

print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {accuracy:.2f}%')

Test Loss: 3.1306, Test Accuracy: 24.39%
Test Loss: 3.1306, Test Accuracy: 24.39%
Test Loss: 3.1306, Test Accuracy: 24.39%
Test Loss: 3.1306, Test Accuracy: 24.39%
Test Loss: 3.1306, Test Accuracy: 24.39%
Test Loss: 3.1306, Test Accuracy: 24.39%
Test Loss: 3.1306, Test Accuracy: 24.39%
Test Loss: 3.1306, Test Accuracy: 24.39%
1min 12s ± 75.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


2. **Testing on CPU**

In [13]:
# Set the device to CPU
device = torch.device("cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5071, 0.4867, 0.4408), std=(0.2675, 0.2565, 0.2761))
])

train_dataset = datasets.CIFAR100(root='./data',
                                  train=True,
                                  download=True,
                                  transform=transform)

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=128,
                          shuffle=True,
                          num_workers=12)

model = models.resnet50(num_classes=100).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Timing the training process
start_time = time.time()

# Training the model
model.train()
total_step = len(train_loader)

for batch_idx, (data, target) in enumerate(train_loader):
    data, target = data.to(device), target.to(device)  
    
    optimizer.zero_grad()
    output = model(data)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()
    
    if batch_idx % 100 == 0:
        print(f'Step [{batch_idx + 1}/{total_step}], Loss: {loss.item():.4f}')

# Calculate and print the time taken for the epoch
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Epoch completed in: {elapsed_time:.2f} seconds on {device}.\n")
print("Training completed.")

Using device: cpu
Files already downloaded and verified
Step [1/391], Loss: 4.9374
Step [101/391], Loss: 4.4203
Step [201/391], Loss: 4.3323
Step [301/391], Loss: 4.0085
Epoch completed in: 1050.41 seconds on cpu.

Training completed.


**Speed up**

Runing the same model on CPU took almost 18 minutes, which is way slower almost 16 times than GPU. 

In [14]:
# Evaluate the model on the test set
model.eval()  

test_dataset = datasets.CIFAR100(root='./data',
                                 train=False,
                                 download=True,
                                 transform=transform)

test_loader = DataLoader(dataset=test_dataset,
                         batch_size=128,
                         shuffle=False,
                         num_workers=4)

correct = 0
total = 0
test_loss = 0.0

with torch.no_grad():  
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)  
        
        outputs = model(data)
        loss = criterion(outputs, target)
        test_loss += loss.item() * data.size(0) 
        
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

avg_loss = test_loss / len(test_loader.dataset)
accuracy = 100 * correct / total

print(f'Test Loss: {avg_loss:.4f}, Accuracy: {accuracy:.2f}%')

Files already downloaded and verified
Test Loss: 7.7852, Accuracy: 12.25%


## Supplementary Information

### Pytorch Available Backends

- Here is the list of the available backends:
    1. cpu
    2. cuda
    3. ipu
    4. xpu
    5. mkldnn
    6. opengl
    7. opencl
    8. ideep
    9. hip
    10. ve
    11. fpga
    12. maia
    13. xla
    14. lazy
    15. vulkan
    16. mps
    17. meta
    18. hpu
    19. mtia
    20. privateuseone

In [16]:
bkend_device_list = [
    "cpu",
    "cuda",
    "ipu",
    "xpu",
    "mkldnn",
    "opengl",
    "opencl",
    "ideep",
    "hip",
    "ve",
    "fpga",
    "maia",
    "xla",
    "lazy",
    "vulkan",
    "mps",
    "meta",
    "hpu",
    "mtia",
    "privateuseone"
]

# Check if each backend is available
for be in bkend_device_list:
    try:
        if hasattr(torch.backends, be) or getattr(torch.backends, be).is_available():
            print(be, ": is available")
    except:
        continue

cpu : is available
cuda : is available
mkldnn : is available
mps : is available


---

### References

1. [PyTorch official documentation](https://pytorch.org/get-started/locally/)
2. [Accelerated PyTorch training on Mac](https://developer.apple.com/metal/pytorch/)
3. [Homebrew](https://brew.sh/)
5. [Miniforge](https://github.com/conda-forge/miniforge)

---